# IMDB-Clean Dataset Preparation (Kaggle)

This notebook processes the `yuulind/imdb-clean` dataset on Kaggle.
It reads the provided CSV, maps the `M`/`F` gender labels to `0`/`1`, crops the faces using the bounding boxes, and saves them to the working directory in the UTKFace format:
`[age]_[gender]_imdb_[id].jpg`.

In [ ]:
import pandas as pd
import cv2
import os
from tqdm.auto import tqdm

# Paths based on Kaggle's yuulind/imdb-clean structure
DATASET_ROOT = '/kaggle/input/datasets/yuulind/imdb-clean'
CSV_PATH = os.path.join(DATASET_ROOT, 'imdb_train_new_1024.csv')
IMAGES_DIR = os.path.join(DATASET_ROOT, 'imdb-clean-1024/imdb-clean-1024')

# Output directory in Kaggle's writable space
OUTPUT_DIR = '/kaggle/working/train_cropped_faces/'

In [ ]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"Loading CSV from {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)
print(f"Total records: {len(df)}")

In [ ]:
def process_imdb_clean(df, images_dir, output_dir):
    successful = 0
    failed = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # The filename in the CSV includes the folder, e.g., '05/nm...'
        img_path = str(row['filename'])
        full_img_path = os.path.join(images_dir, img_path)
        
        img = cv2.imread(full_img_path)
        if img is None:
            failed += 1
            continue
            
        # Parse Gender: M -> 0, F -> 1
        gender_str = str(row['gender']).strip().upper()
        if gender_str == 'M':
            gender_idx = 0
        elif gender_str == 'F':
            gender_idx = 1
        else:
            failed += 1
            continue
            
        # Parse Age
        try:
            age = int(float(row['age']))
        except (ValueError, TypeError):
            failed += 1
            continue
            
        if age < 0 or age > 116:
            failed += 1
            continue
            
        # Extract bounding box coordinates
        try:
            x_min, y_min = int(row['x_min']), int(row['y_min'])
            x_max, y_max = int(row['x_max']), int(row['y_max'])
        except (ValueError, TypeError):
            failed += 1
            continue
            
        # Ensure bounding box is within image bounds
        h, w, _ = img.shape
        x_min, y_min = max(0, x_min), max(0, y_min)
        x_max, y_max = min(w, x_max), min(h, y_max)
        
        # Ensure the crop is valid (has positive width/height)
        if x_max <= x_min or y_max <= y_min:
            failed += 1
            continue
            
        # Crop face
        face_crop = img[y_min:y_max, x_min:x_max]
        
        # UTKFace Format: [age]_[gender]_[race]_[date].jpg
        # 'imdb' replaces the race tag.
        new_filename = f"{age}_{gender_idx}_imdb_{idx}.jpg"
        
        cv2.imwrite(os.path.join(output_dir, new_filename), face_crop)
        successful += 1
        
    print(f"\nCompleted! Successfully processed: {successful}, Failed/Skipped: {failed}")

# Start processing
process_imdb_clean(df, IMAGES_DIR, OUTPUT_DIR)

## Check the Results
Visualize a few of the cropped images to verify the `age` and `gender` parsing is flawless.

In [ ]:
import matplotlib.pyplot as plt
import glob

cropped_files = glob.glob(os.path.join(OUTPUT_DIR, "*.jpg"))
if cropped_files:
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    for i in range(min(5, len(cropped_files))):
        img_path = cropped_files[i]
        basename = os.path.basename(img_path)
        
        # Parse back the UTKFace format
        parts = basename.split('_')
        age = parts[0]
        gender = "Male" if parts[1] == '0' else "Female"
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(img)
        axes[i].set_title(f"Age: {age} | Gender: {gender}")
        axes[i].axis('off')
    plt.show()